# Homework 3
-   **Name:**  Victor Hugo Gomez Soto 
-  **e-mail:** -- victor.gomez2701@alumnos.udg.mx --


# MODULES

In [96]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


# Activity 1: Path length - BM1 vs BM2 vs CRW

In [97]:

# 1. Brownian Motion (BM) Trajectory with Fixed Step Size
def brownian_motion(n_steps, step_size=1):
    angles = np.random.uniform(0, 2*np.pi, n_steps)
    steps = np.column_stack((step_size * np.cos(angles), step_size * np.sin(angles)))
    trajectory = np.cumsum(steps, axis=0)
    return pd.DataFrame(trajectory, columns=['x', 'y'])

# 2. Correlated Random Walk (CRW) with High Correlation
def correlated_random_walk(n_steps, correlation=50):
    angles = np.cumsum(np.random.vonmises(0, correlation, n_steps))
    steps = np.column_stack((np.cos(angles), np.sin(angles)))
    trajectory = np.cumsum(steps, axis=0)
    return pd.DataFrame(trajectory, columns=['x', 'y'])

# 3. Path Length Calculation (Cumulative)
def cumulative_path_length(traj):
    step_lengths = np.sqrt(np.diff(traj['x'])**2 + np.diff(traj['y'])**2)
    return np.cumsum(step_lengths)

# Generate trajectories with same values as PDF reference
bm_traj1 = brownian_motion(1000, step_size=1)
bm_traj2 = brownian_motion(1000, step_size=1.2)
crw_traj = correlated_random_walk(1000, correlation=30)

# Compute cumulative path lengths
bm_length1 = cumulative_path_length(bm_traj1)
bm_length2 = cumulative_path_length(bm_traj2)
crw_length = cumulative_path_length(crw_traj)

# Time steps
time_steps = np.arange(1, len(bm_length1) + 1)

# Plot Path Length Comparison to match PDF style exactly
fig_path = go.Figure()
fig_path.add_trace(go.Scatter(x=time_steps, y=bm_length1, mode='lines', line=dict(width=2, color='blue'), name='Path Length BM 3'))
fig_path.add_trace(go.Scatter(x=time_steps, y=bm_length2, mode='lines', line=dict(width=2, color='red'), name='Path Length BM 6'))
fig_path.add_trace(go.Scatter(x=time_steps, y=crw_length, mode='lines', line=dict(width=2, color='green'), name='Path Length CRW 6'))
fig_path.update_layout(title='Cumulative Path Length Comparison', xaxis_title='Time Steps', yaxis_title='Path Length', legend=dict(x=1, y=1))
fig_path.show()





# Activity 2: Lévy Distribution - N Different Curves

In [98]:


# 1. Brownian Motion (BM) Trajectory with Fixed Step Size
def brownian_motion(n_steps, step_size=1):
    angles = np.random.uniform(0, 2*np.pi, n_steps)
    steps = np.column_stack((step_size * np.cos(angles), step_size * np.sin(angles)))
    trajectory = np.cumsum(steps, axis=0)
    return pd.DataFrame(trajectory, columns=['x', 'y'])

# 2. Correlated Random Walk (CRW) with Stronger Correlation
def correlated_random_walk(n_steps, correlation=0.999, step_size=5):
    angles = np.cumsum(np.random.vonmises(0, correlation, n_steps))
    steps = np.column_stack((step_size * np.cos(angles), step_size * np.sin(angles)))
    trajectory = np.cumsum(steps, axis=0)
    return pd.DataFrame(trajectory, columns=['x', 'y'])

# 3. Mean Squared Displacement (MSD) Calculation
def mean_squared_displacement(traj):
    msd = np.array([(traj.iloc[i]['x'] - traj.iloc[0]['x'])**2 + (traj.iloc[i]['y'] - traj.iloc[0]['y'])**2 for i in range(len(traj))])
    return msd

# Generate trajectories with adjusted values
bm_traj = brownian_motion(1000, step_size=2)  # BM 6
crw_traj = correlated_random_walk(1000, correlation=0.999, step_size=5)  # CRW 6 with stronger correlation

# Compute MSD
bm_msd = mean_squared_displacement(bm_traj)
crw_msd = mean_squared_displacement(crw_traj)

# Apply a moving average to smooth the CRW MSD
def moving_average(data, window_size=20):
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

crw_msd_smoothed = moving_average(crw_msd, window_size=50)
bm_msd_smoothed = moving_average(bm_msd, window_size=50)

# Time steps (adjusted for moving average window)
time_steps = np.arange(len(crw_msd_smoothed))

# Plot MSD Comparison to match PDF style
fig_msd = go.Figure()
fig_msd.add_trace(go.Scatter(x=time_steps, y=bm_msd_smoothed, mode='lines', line=dict(width=2, color='blue'), name='MSD BM 6'))
fig_msd.add_trace(go.Scatter(x=time_steps, y=crw_msd_smoothed, mode='lines', line=dict(width=2, color='red'), name='MSD CRW 6 c=0.9'))
fig_msd.update_layout(title='Mean Squared Displacement Comparison', xaxis_title='Time Steps', yaxis_title='MSD', legend=dict(x=1, y=1))
fig_msd.show()



# Activity 3: Histograms + Curves


In [99]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import cauchy

# 1. Correlated Random Walk (CRW) with Cauchy-distributed Turning Angles
def correlated_random_walk_cauchy(n_steps, cauchy_scale, step_size=1):
    angles = np.cumsum(cauchy.rvs(scale=cauchy_scale, size=n_steps))
    steps = np.column_stack((step_size * np.cos(angles), step_size * np.sin(angles)))
    trajectory = np.cumsum(steps, axis=0)
    return pd.DataFrame(trajectory, columns=['x', 'y']), angles

# 2. Compute Turning Angles
def compute_turning_angles(angles):
    turning_angles = np.diff(angles)
    turning_angles = np.mod(turning_angles + np.pi, 2*np.pi) - np.pi  # Normalize to [-π, π]
    return turning_angles

# 3. Generate CRW Trajectories with Different Cauchy Coefficients
crw_traj_06, angles_06 = correlated_random_walk_cauchy(1000, cauchy_scale=0.6, step_size=2)
crw_traj_09, angles_09 = correlated_random_walk_cauchy(1000, cauchy_scale=0.9, step_size=2)

# 4. Compute Observed Turning Angles
turning_angles_06 = compute_turning_angles(angles_06)
turning_angles_09 = compute_turning_angles(angles_09)

# 5. Generate Theoretical Cauchy Distributions
x_vals = np.linspace(-np.pi, np.pi, 1000)
cauchy_pdf_06 = cauchy.pdf(x_vals, scale=0.6)
cauchy_pdf_09 = cauchy.pdf(x_vals, scale=0.9)

# 6. Plot Turning-angle Distribution (Observed vs Theoretical)
fig_turning = go.Figure()
fig_turning.add_trace(go.Histogram(x=turning_angles_06, histnorm='probability density', nbinsx=500, name='Observed 0.6', opacity=0.5, marker_color='blue'))
fig_turning.add_trace(go.Histogram(x=turning_angles_09, histnorm='probability density', nbinsx=500, name='Observed 0.9', opacity=0.5, marker_color='red'))
fig_turning.add_trace(go.Scatter(x=x_vals, y=cauchy_pdf_06, mode='lines', line=dict(width=2, color='blue'), name='Cauchy 0.6'))
fig_turning.add_trace(go.Scatter(x=x_vals, y=cauchy_pdf_09, mode='lines', line=dict(width=2, color='red'), name='Cauchy 0.9'))
fig_turning.update_layout(title='Turning-angle Distribution (Observed vs Theoretical)', xaxis_title='Turning Angle (radians)', yaxis_title='Probability Density', barmode='overlay')
fig_turning.show()



# Activity 4: Lévy Flight - Vec2d - 1 Trajectory

In [102]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import levy_stable

# 1. Lévy Walk (LW) Trajectory with Alpha Coefficient
def levy_walk(n_steps, alpha, beta=0, step_size=1, max_step=50):
    step_lengths = levy_stable.rvs(alpha, beta, size=n_steps) * step_size
    step_lengths = np.clip(step_lengths, 0, max_step)  # Limit step size to match PDF image scale
    step_lengths -= step_lengths[0]  # Ensure it starts at zero
    angles = np.random.uniform(0, 2*np.pi, n_steps)
    angles_shifted = angles + np.pi / 2  # Apply 90-degree phase shift
    steps = np.column_stack((step_lengths * np.cos(angles_shifted), step_lengths * np.sin(angles_shifted)))
    trajectory = np.cumsum(steps, axis=0)
    return pd.DataFrame(trajectory, columns=['x', 'y']), step_lengths

# 2. Generate Lévy Walks with Different Alpha Coefficients
lw_traj_10, step_lengths_10 = levy_walk(1000, alpha=1.0, beta=0, step_size=2, max_step=50)
lw_traj_07, step_lengths_07 = levy_walk(1000, alpha=0.7, beta=0, step_size=2, max_step=50)

# 3. Generate Theoretical Lévy Distributions
x_vals = np.linspace(0, 50, 1000)
levy_pdf_10 = levy_stable.pdf(x_vals, alpha=1.0, beta=0)
levy_pdf_07 = levy_stable.pdf(x_vals, alpha=0.7, beta=0)

# 4. Plot Step-length Distribution (Observed vs Theoretical)
fig_step_length = go.Figure()
fig_step_length.add_trace(go.Histogram(x=step_lengths_10, histnorm='probability density', nbinsx=500, name='Observed_alpha=1.0_beta=0', opacity=0.5, marker_color='blue'))
fig_step_length.add_trace(go.Histogram(x=step_lengths_07, histnorm='probability density', nbinsx=500, name='Observed_alpha=0.7_beta=0', opacity=0.5, marker_color='purple'))
fig_step_length.add_trace(go.Scatter(x=x_vals, y=levy_pdf_10, mode='lines', line=dict(width=2, color='blue'), name='Levy_alpha=1.0'))
fig_step_length.add_trace(go.Scatter(x=x_vals, y=levy_pdf_07, mode='lines', line=dict(width=2, color='red'), name='Levy_alpha=0.7'))
fig_step_length.update_layout(title='Step-length Distribution (Observed vs Theoretical)', xaxis_title='Step Length', yaxis_title='Probability Density', barmode='overlay')
fig_step_length.show()

